# Step 16c: What-If Policy Simulation Engine

## Overview
This notebook implements the "What-If" HR policy simulation engine:
- Takes a single employee's feature row.
- Applies hypothetical policy perturbations (e.g. +10% salary hike, removing overtime, improving work-life balance).
- Recalculates `pipeline.predict_proba()` without retraining the model.
- Provides immediate feedback on predicted risk reduction (later exposed in Day 4 FastAPI `/simulate/whatif` endpoint and Streamlit dashboard panel).


In [1]:
import pandas as pd
import numpy as np
import os
import joblib

PROCESSED_DIR = os.path.join("..", "data", "processed")
MODELS_DIR = os.path.join("..", "models")

features_df = pd.read_csv(os.path.join(PROCESSED_DIR, "features_engineered.csv"))
pipeline = joblib.load(os.path.join(MODELS_DIR, "attrition_pipeline.joblib"))

drop_cols = ['EmployeeNumber', 'Employee ID', 'Attrition', 'Target_Attrition']
feature_cols = [c for c in features_df.columns if c not in drop_cols]


---
## 1. What-If Simulation Engine Function


In [2]:
def simulate_employee_policy(emp_id: int, overrides: dict):
    # Locate employee row
    emp_rows = features_df[features_df['EmployeeNumber'] == emp_id]
    if len(emp_rows) == 0:
        return {"error": f"Employee {emp_id} not found."}
    
    emp_data = emp_rows.iloc[0].copy()
    
    # Baseline prediction
    base_row = pd.DataFrame([emp_data[feature_cols]])
    base_prob = pipeline.predict_proba(base_row)[0, 1]
    
    # Apply overrides
    sim_data = emp_data.copy()
    for key, val in overrides.items():
        if key in sim_data:
            sim_data[key] = val
            
    # Recalculate dynamic engineered features if raw inputs changed
    if 'MonthlyIncome' in overrides or 'YearsAtCompany' in overrides:
        sim_data['Income_Per_Company_Year'] = sim_data['MonthlyIncome'] / (sim_data['YearsAtCompany'] + 1.0)
        
    if 'YearsSinceLastPromotion' in overrides or 'YearsInCurrentRole' in overrides:
        sim_data['Promotion_Delay_Ratio'] = sim_data['YearsSinceLastPromotion'] / (sim_data['YearsInCurrentRole'] + 1.0)
        
    if any(k in overrides for k in ['EnvironmentSatisfaction', 'JobSatisfaction', 'RelationshipSatisfaction', 'WorkLifeBalance']):
        sim_data['Overall_Satisfaction_Index'] = (sim_data['EnvironmentSatisfaction'] + sim_data['JobSatisfaction'] + 
                                                   sim_data['RelationshipSatisfaction'] + sim_data['WorkLifeBalance']) / 4.0

    sim_row = pd.DataFrame([sim_data[feature_cols]])
    new_prob = pipeline.predict_proba(sim_row)[0, 1]
    
    delta_prob = new_prob - base_prob
    pct_change = (delta_prob / base_prob) * 100.0 if base_prob > 0 else 0.0
    
    return {
        "EmployeeNumber": emp_id,
        "Baseline_Attrition_Risk": round(float(base_prob), 4),
        "Simulated_Attrition_Risk": round(float(new_prob), 4),
        "Risk_Difference": round(float(delta_prob), 4),
        "Percentage_Risk_Reduction": round(float(pct_change), 2),
        "Overrides_Applied": overrides
    }

# Demonstration on Sample Employee
sample_emp = features_df[features_df['OverTime'] == 'Yes']['EmployeeNumber'].iloc[0]
print(f"Running What-If Simulation for Sample Employee #{sample_emp}:")

test_overrides = {
    'OverTime': 'No',
    'MonthlyIncome': features_df[features_df['EmployeeNumber'] == sample_emp]['MonthlyIncome'].iloc[0] * 1.15,
    'WorkLifeBalance': 4
}

res = simulate_employee_policy(sample_emp, test_overrides)
for k, v in res.items():
    print(f"  {k}: {v}")


Running What-If Simulation for Sample Employee #1:
  EmployeeNumber: 1
  Baseline_Attrition_Risk: 0.9013
  Simulated_Attrition_Risk: 0.3812
  Risk_Difference: -0.5201
  Percentage_Risk_Reduction: -57.71
  Overrides_Applied: {'OverTime': 'No', 'MonthlyIncome': np.float64(6891.95), 'WorkLifeBalance': 4}
